# Classification binaire d'images

Notebook de suivi du projet : TP1 CNN scratch, TP2 augmentation + Dropout, TP3 transfer learning MobileNetV2.

## Phase 1.1 - Setup Colab et organisation du dataset

Objectif : configurer Colab, telecharger le dataset Kaggle, organiser les images en `train/val` par classe, puis afficher quelques exemples pour verifier le split.

In [ ]:
# === CONFIGURATION DU PROJET ===
# Dataset Kaggle : https://www.kaggle.com/datasets/chetankv/dogs-cats-images
# Ces constantes sont le seul endroit a modifier pour changer de sujet.

CLASS_A = "cat"
CLASS_B = "dog"

DATA_ROOT = "/content/data"
RAW_ROOT = f"{DATA_ROOT}/raw"
KAGGLE_DATASET = "chetankv/dogs-cats-images"
KAGGLE_ZIP = "dogs-cats-images.zip"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SEED = 42
TRAIN_RATIO = 0.80

### 1.1.1 - Installation et authentification Kaggle

Dans Colab, telecharger `kaggle.json` depuis Kaggle : Settings > API > Create New Token.

In [ ]:
!pip install kaggle -q

In [ ]:
import os
from google.colab import files

uploaded = files.upload()  # selectionner kaggle.json

if "kaggle.json" not in uploaded:
    raise FileNotFoundError("Le fichier kaggle.json doit etre selectionne.")

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
if os.path.exists(kaggle_json_path):
    os.remove(kaggle_json_path)

os.rename("kaggle.json", kaggle_json_path)
os.chmod(kaggle_json_path, 0o600)

print(f"Token Kaggle installe : {kaggle_json_path}")
!ls -la ~/.kaggle/

### 1.1.2 - Telechargement et extraction du dataset

In [ ]:
import os

os.makedirs(RAW_ROOT, exist_ok=True)
!kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_ROOT}
!unzip -q -o {RAW_ROOT}/{KAGGLE_ZIP} -d {RAW_ROOT}

print("Extraction terminee dans", RAW_ROOT)

In [ ]:
from pathlib import Path

for path in sorted(Path(RAW_ROOT).glob("**/*"))[:40]:
    print(path.relative_to(RAW_ROOT))

### 1.1.3 - Organisation en train/val

La structure finale attendue est `DATA_ROOT/train/cat`, `DATA_ROOT/train/dog`, `DATA_ROOT/val/cat`, `DATA_ROOT/val/dog`.

In [ ]:
import random
import shutil
from pathlib import Path


def is_image(path):
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def collect_images_for_class(raw_root, class_name):
    raw_root = Path(raw_root)
    singular = class_name.lower()
    plural = f"{singular}s"

    class_dirs = [
        path for path in raw_root.glob("**/*")
        if path.is_dir() and path.name.lower() in {singular, plural}
    ]

    files = []
    for class_dir in class_dirs:
        files.extend(path for path in class_dir.rglob("*") if is_image(path))

    unique_files = sorted(set(files))
    if not unique_files:
        raise ValueError(f"Aucune image trouvee pour la classe {class_name} dans {raw_root}")

    return unique_files


files_a = collect_images_for_class(RAW_ROOT, CLASS_A)
files_b = collect_images_for_class(RAW_ROOT, CLASS_B)

print(CLASS_A, len(files_a), "images brutes")
print(CLASS_B, len(files_b), "images brutes")

In [ ]:
import os
import random
import shutil


def reset_output_dirs(data_root, classes):
    for split in ["train", "val"]:
        for cls in classes:
            path = os.path.join(data_root, split, cls)
            if os.path.exists(path):
                shutil.rmtree(path)
            os.makedirs(path, exist_ok=True)


def split_files(files, seed=SEED, train_ratio=TRAIN_RATIO):
    files = list(files)
    rng = random.Random(seed)
    rng.shuffle(files)
    split_index = int(len(files) * train_ratio)
    return files[:split_index], files[split_index:]


def copy_split(files, split, class_name):
    target_dir = os.path.join(DATA_ROOT, split, class_name)
    for source_path in files:
        target_path = os.path.join(target_dir, source_path.name)
        if os.path.exists(target_path):
            stem, suffix = source_path.stem, source_path.suffix
            target_path = os.path.join(target_dir, f"{stem}_{abs(hash(str(source_path))) % 10**8}{suffix}")
        shutil.copy2(source_path, target_path)


reset_output_dirs(DATA_ROOT, [CLASS_A, CLASS_B])

train_a, val_a = split_files(files_a)
train_b, val_b = split_files(files_b)

copy_split(train_a, "train", CLASS_A)
copy_split(val_a, "val", CLASS_A)
copy_split(train_b, "train", CLASS_B)
copy_split(val_b, "val", CLASS_B)

print("Organisation terminee.")

### 1.1.4 - Verification des comptes

In [ ]:
counts = {}

for split in ["train", "val"]:
    for cls in [CLASS_A, CLASS_B]:
        path = os.path.join(DATA_ROOT, split, cls)
        image_count = len([name for name in os.listdir(path) if Path(name).suffix.lower() in IMAGE_EXTENSIONS])
        counts[(split, cls)] = image_count
        print(f"{path} : {image_count} images")

for cls in [CLASS_A, CLASS_B]:
    total = counts[("train", cls)] + counts[("val", cls)]
    train_ratio = counts[("train", cls)] / total
    print(f"{cls} : ratio train={train_ratio:.2%}, total={total}")
    assert counts[("train", cls)] >= 500, f"Pas assez d'images train pour {cls}"
    assert abs(train_ratio - TRAIN_RATIO) <= 0.01, f"Ratio train/val incorrect pour {cls}"

print("Verification des comptes OK.")

### 1.1.5 - Verification visuelle

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt


def sample_images(split, class_name, n=3):
    class_dir = Path(DATA_ROOT) / split / class_name
    images = sorted(path for path in class_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
    return images[:n]


samples = [(CLASS_A, path) for path in sample_images("train", CLASS_A, 3)]
samples += [(CLASS_B, path) for path in sample_images("train", CLASS_B, 3)]

plt.figure(figsize=(12, 6))
for i, (class_name, image_path) in enumerate(samples):
    image = mpimg.imread(image_path)
    print(f"{image_path.name} | {class_name} | shape={image.shape}")

    if len(image.shape) == 3:
        assert image.shape[-1] in [1, 3, 4], f"Canaux inattendus pour {image_path}: {image.shape}"
    else:
        assert len(image.shape) == 2, f"Shape inattendue pour {image_path}: {image.shape}"

    plt.subplot(2, 3, i + 1)
    plt.imshow(image, cmap="gray" if len(image.shape) == 2 else None)
    plt.title(class_name)
    plt.axis("off")

plt.tight_layout()
plt.show()

### 1.1.6 - Edge case : seed different, ratio stable

Changer le seed modifie les fichiers selectionnes, mais pas les comptes par split sauf effet d'arrondi.

In [ ]:
def split_count_summary(files, seed):
    train_files, val_files = split_files(files, seed=seed)
    return len(train_files), len(val_files)


for cls, files in [(CLASS_A, files_a), (CLASS_B, files_b)]:
    train_42, val_42 = split_count_summary(files, seed=42)
    train_0, val_0 = split_count_summary(files, seed=0)
    print(f"{cls} | seed=42 : train={train_42}, val={val_42}")
    print(f"{cls} | seed=0  : train={train_0}, val={val_0}")
    assert abs(train_42 - train_0) <= 1
    assert abs(val_42 - val_0) <= 1

print("Verification seed OK.")

## Phase 1.2 - Preprocessing : normalisation et batching

Objectif : charger les images avec `image_dataset_from_directory`, normaliser les pixels, configurer le batching et verifier les shapes du premier batch.

In [ ]:
import os
import tensorflow as tf

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_path = os.path.join(DATA_ROOT, "train")
val_path = os.path.join(DATA_ROOT, "val")

print("Dossiers train :", sorted(os.listdir(train_path)))
print("Dossiers val   :", sorted(os.listdir(val_path)))

expected_classes = sorted([CLASS_A, CLASS_B])
assert sorted(os.listdir(train_path)) == expected_classes, "Dossier parasite ou classe manquante dans train/"
assert sorted(os.listdir(val_path)) == expected_classes, "Dossier parasite ou classe manquante dans val/"

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,
)

print("Classes detectees :", train_ds.class_names)

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1.0 / 255)

train_ds = train_ds.map(lambda images, labels: (normalization_layer(images), labels))
val_ds = val_ds.map(lambda images, labels: (normalization_layer(images), labels))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
batch_images, batch_labels = next(iter(train_ds))

print("Shape images batch :", batch_images.shape)
print("Shape labels batch :", batch_labels.shape)
print(f"Valeurs pixels : min={tf.reduce_min(batch_images).numpy():.2f}, max={tf.reduce_max(batch_images).numpy():.2f}")

assert batch_images.shape == (BATCH_SIZE, IMG_SIZE[0], IMG_SIZE[1], 3)
assert batch_labels.shape == (BATCH_SIZE, 1)
assert float(tf.reduce_min(batch_images)) >= 0.0
assert float(tf.reduce_max(batch_images)) <= 1.0

print("Pipeline dataset OK.")

Note edge case : sans `.cache()` sur `train_ds`, le pipeline relit les images depuis le disque a chaque epoch et l'entrainement devient nettement plus lent.

## Phase 1.3 - Architecture CNN from scratch

Objectif : construire et compiler un CNN simple from scratch, verifier les shapes du `model.summary()` et identifier le nombre de parametres.

### Calculs avant execution

Avec `IMG_SIZE = (128, 128)` et `padding='same'` :

- Bloc 1 apres MaxPooling : `(64, 64, 32)`
- Bloc 2 apres MaxPooling : `(32, 32, 64)`
- Bloc 3 apres MaxPooling : `(16, 16, 128)`
- Flatten : `16 * 16 * 128 = 32768` valeurs
- Dense(128) : `(32768 + 1) * 128 = 4194432` parametres

La sortie finale attendue est `(None, 1)` avec une activation `sigmoid`.

In [ ]:
from tensorflow.keras import layers, models


def build_cnn_scratch(input_shape):
    """
    CNN from scratch pour la classification binaire.
    Architecture deliberement simple : on veut voir l'overfitting, pas l'eviter.
    input_shape : tuple (H, W, C) correspondant a IMG_SIZE + nombre de canaux.
    Retourne : un modele Sequential non compile.
    """
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    return model


model_scratch = build_cnn_scratch(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
model_scratch.summary()

In [ ]:
flatten_units = (IMG_SIZE[0] // 8) * (IMG_SIZE[1] // 8) * 128
dense_128_params = (flatten_units + 1) * 128

print("Flatten attendu :", flatten_units)
print("Parametres attendus Dense(128) :", dense_128_params)

assert flatten_units == 32768
assert dense_128_params == 4_194_432
assert model_scratch.output_shape == (None, 1)
assert model_scratch.layers[-1].activation.__name__ == "sigmoid"

In [ ]:
model_scratch.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

print("Modele CNN scratch compile.")

Note edge case : avec `padding='valid'`, chaque convolution reduit la hauteur et la largeur avant le pooling. Le Flatten contient donc moins d'elements et la couche `Dense(128)` a moins de parametres.

Note adversarial : la derniere couche doit rester `Dense(1, activation='sigmoid')` pour produire une probabilite compatible avec `binary_crossentropy`.

## Phase 1.4 - Entrainement et diagnostic overfitting

Objectif : entrainer le CNN scratch pendant 20 epochs, logger TensorBoard, visualiser les courbes train/val et identifier le point de divergence.

In [ ]:
import datetime
import time

log_dir = "logs/scratch/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

print("TensorBoard log_dir :", log_dir)

In [ ]:
start = time.time()

history_scratch = model_scratch.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=[tensorboard_callback, early_stopping],
)

training_time_scratch = time.time() - start

print(f"Temps d'entrainement : {training_time_scratch:.0f}s")
print(f"val_accuracy finale : {max(history_scratch.history['val_accuracy']):.3f}")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/scratch

### Courbes loss/accuracy

`plot_history` est l'utilitaire commun a reutiliser pour comparer les modeles des phases suivantes.

In [ ]:
import matplotlib.pyplot as plt


def plot_history(history, title=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history["loss"], label="Train loss")
    ax1.plot(history.history["val_loss"], label="Val loss")
    ax1.set_title(f"{title} - Loss")
    ax1.set_xlabel("Epoch")
    ax1.legend()

    ax2.plot(history.history["accuracy"], label="Train accuracy")
    ax2.plot(history.history["val_accuracy"], label="Val accuracy")
    ax2.set_title(f"{title} - Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.legend()

    plt.tight_layout()
    plt.savefig(f"curves_{title.lower().replace(' ', '_')}.png", dpi=100)
    plt.show()


plot_history(history_scratch, "CNN scratch")

In [ ]:
best_val_loss_epoch = int(min(range(len(history_scratch.history["val_loss"])), key=history_scratch.history["val_loss"].__getitem__)) + 1
best_val_loss = min(history_scratch.history["val_loss"])
best_val_accuracy = max(history_scratch.history["val_accuracy"])

print(f"Meilleure val_loss : {best_val_loss:.4f} a l'epoch {best_val_loss_epoch}")
print(f"Meilleure val_accuracy : {best_val_accuracy:.3f}")
print("Point de divergence a commenter : premiere epoch ou val_loss remonte durablement pendant que train_loss baisse.")

Diagnostic attendu : la `train_accuracy` continue de monter alors que la `val_accuracy` plafonne, et la `val_loss` remonte apres quelques epochs. C'est l'overfitting recherche pour le CNN scratch.

Edge case : avec `BATCH_SIZE = 4`, le gradient devient plus bruite et les courbes peuvent osciller fortement. Comparer la meilleure `val_accuracy` avec celle obtenue en batch 32.

Adversarial : si `shuffle=True` est oublie dans `image_dataset_from_directory`, les batches peuvent contenir des images quasi toutes de la meme classe, ce qui rend l'apprentissage instable.

## Phase 2.1 - Pipeline d'augmentation

Objectif : integrer des couches d'augmentation Keras et verifier visuellement les transformations avant l'entrainement.

Les couches d'augmentation sont actives uniquement en mode training (`model.fit`) et passives en inference (`model.predict`, `model.evaluate`). La `val_accuracy` restera donc calculee sur les images originales, ce qui est voulu.

In [ ]:
from tensorflow.keras import layers, models

data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
], name="data_augmentation")

data_augmentation.summary()

In [ ]:
sample_batch_images, sample_batch_labels = next(iter(train_ds))
sample_image = sample_batch_images[0]

plt.figure(figsize=(8, 8))
for i in range(9):
    augmented = data_augmentation(tf.expand_dims(sample_image, 0), training=True)
    plt.subplot(3, 3, i + 1)
    plt.imshow(augmented[0])
    plt.axis("off")

plt.tight_layout()
plt.savefig("augmentation_grid.png", dpi=100)
plt.show()

Checkpoint visuel : les 9 images doivent rester reconnaissables comme la meme classe. Pour des chats/chiens, `RandomFlip("horizontal")`, `RandomRotation(0.08)` et `RandomZoom(0.10)` sont realistes.

Edge case : empiler plusieurs rotations fortes rend les images meconnaissables. Plus d'augmentation n'est pas automatiquement mieux.

Adversarial : `RandomFlip("vertical")` est contre-productif pour des photos naturelles orientees, car les animaux tete en bas ne correspondent pas au test set attendu.

## Phase 2.2 - Dropout et reentrainement

Objectif : construire un CNN augmente avec Dropout, le reentrainer depuis zero et comparer les courbes avec le CNN scratch.

In [ ]:
def build_cnn_augmented(input_shape):
    """
    CNN avec data augmentation integree + Dropout.
    Meme architecture de base que build_cnn_scratch, mais avec regularisation.
    input_shape : tuple (H, W, C).
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        data_augmentation,

        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dropout(0.4),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    return model


model_augmented = build_cnn_augmented(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
model_augmented.summary()

In [ ]:
model_augmented.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

assert model_augmented.output_shape == (None, 1)
assert model_augmented.layers[-1].activation.__name__ == "sigmoid"
assert any(isinstance(layer, layers.Dropout) and layer.rate == 0.4 for layer in model_augmented.layers)

print("Modele CNN augmente + Dropout compile.")

In [ ]:
log_dir_aug = "logs/augmented/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback_aug = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir_aug,
    histogram_freq=1,
)

early_stopping_aug = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

print("TensorBoard log_dir augmented :", log_dir_aug)

In [ ]:
start = time.time()

history_augmented = model_augmented.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=[tensorboard_callback_aug, early_stopping_aug],
)

training_time_augmented = time.time() - start

print(f"Temps d'entrainement : {training_time_augmented:.0f}s")
print(f"val_accuracy finale : {max(history_augmented.history['val_accuracy']):.3f}")

In [ ]:
%tensorboard --logdir logs/augmented

In [ ]:
plot_history(history_augmented, "CNN augmente + Dropout")

In [ ]:
best_scratch_val_accuracy = max(history_scratch.history["val_accuracy"])
best_augmented_val_accuracy = max(history_augmented.history["val_accuracy"])
scratch_gap = max(history_scratch.history["accuracy"]) - best_scratch_val_accuracy
augmented_gap = max(history_augmented.history["accuracy"]) - best_augmented_val_accuracy

print(f"Best val_accuracy scratch   : {best_scratch_val_accuracy:.3f}")
print(f"Best val_accuracy augmente  : {best_augmented_val_accuracy:.3f}")
print(f"Gap train/val scratch       : {scratch_gap:.3f}")
print(f"Gap train/val augmente      : {augmented_gap:.3f}")

Diagnostic attendu : la `val_accuracy` monte plus haut que le CNN scratch et le gap train/val diminue. La convergence peut etre plus lente, car le modele memorise moins facilement le train set.

Edge case : placer `Dropout` avant `Flatten` coupe les feature maps spatiales et peut degrader la `val_accuracy`. Ici, il est volontairement place apres `Flatten`.

Adversarial : `Dropout(0.8)` injecte trop de bruit ; le modele risque de ne plus apprendre. Un range raisonnable pour ce CNN est `0.2` a `0.4` apres `Flatten`.

## Phase 2.3 - Comparaison TP1 vs TP2

Objectif : comparer systematiquement les courbes du CNN scratch et du CNN augmente + Dropout, mettre les chiffres en regard et interpreter l'effet de la regularisation.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_scratch.history["val_loss"], color="red", label="Scratch val_loss")
plt.plot(history_augmented.history["val_loss"], color="blue", label="Augmente + Dropout val_loss")
plt.title("Validation loss TP1 vs TP2")
plt.xlabel("Epoch")
plt.ylabel("val_loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_scratch.history["val_accuracy"], color="red", label="Scratch val_accuracy")
plt.plot(history_augmented.history["val_accuracy"], color="blue", label="Augmente + Dropout val_accuracy")
plt.title("Validation accuracy TP1 vs TP2")
plt.xlabel("Epoch")
plt.ylabel("val_accuracy")
plt.legend()

plt.tight_layout()
plt.savefig("comparison_tp1_tp2.png", dpi=100)
plt.show()

In [ ]:
import pandas as pd

comparison_tp1_tp2 = {
    "modele": ["CNN scratch", "CNN augmente + Dropout"],
    "best_val_accuracy": [
        max(history_scratch.history["val_accuracy"]),
        max(history_augmented.history["val_accuracy"]),
    ],
    "best_val_loss": [
        min(history_scratch.history["val_loss"]),
        min(history_augmented.history["val_loss"]),
    ],
    "training_time_s": [training_time_scratch, training_time_augmented],
    "params": [model_scratch.count_params(), model_augmented.count_params()],
    "gap_train_val_accuracy": [scratch_gap, augmented_gap],
    "epochs_run": [
        len(history_scratch.history["loss"]),
        len(history_augmented.history["loss"]),
    ],
}

comparison_tp1_tp2_df = pd.DataFrame(comparison_tp1_tp2)
comparison_tp1_tp2_df

In [ ]:
val_accuracy_gain = best_augmented_val_accuracy - best_scratch_val_accuracy
gap_reduction = scratch_gap - augmented_gap
time_delta = training_time_augmented - training_time_scratch

print(f"Gain best val_accuracy TP2 - TP1 : {val_accuracy_gain:.3f}")
print(f"Reduction du gap train/val       : {gap_reduction:.3f}")
print(f"Difference temps entrainement    : {time_delta:.0f}s")

if gap_reduction > 0:
    print("Le gap train/val est reduit avec l'augmentation + Dropout.")
else:
    print("Le gap train/val n'est pas reduit sur ce run : verifier les courbes et relancer avec plusieurs seeds.")

Interpretation TP1 vs TP2 : apres execution, le CNN augmente + Dropout doit montrer une courbe de validation plus stable que le CNN scratch. Le gap train/val se mesure avec `scratch_gap` et `augmented_gap` ; si `gap_reduction` est positif, la regularisation a bien attenue l'overfitting. La convergence peut etre plus lente en TP2, car chaque batch contient des variations aleatoires et le Dropout empeche le modele de memoriser trop vite les exemples d'entrainement. L'augmentation seule ne suffit pas toujours : elle ne remplace pas des features plus robustes, ni davantage de donnees variees, et un CNN entraine from scratch reste limite par la taille du dataset et par sa capacite a apprendre des representations generales. Pour conclure honnetement, il faut verifier que le gain TP2 depasse la variance observee entre plusieurs seeds.

## Phase 3.1 - Chargement MobileNetV2 et freezing

Objectif : charger MobileNetV2 pre-entraine sur ImageNet, geler la base, construire une tete de classification custom et verifier que seuls les parametres de la tete sont entrainables.

MobileNetV2 attend des images d'au moins 32x32. Pour le transfer learning, on reconstruit le pipeline en `160x160` afin d'utiliser des representations plus riches que le pipeline scratch en `128x128`.

In [ ]:
IMG_SIZE_TL = (160, 160)

train_ds_tl = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=IMG_SIZE_TL,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

val_ds_tl = tf.keras.utils.image_dataset_from_directory(
    val_path,
    image_size=IMG_SIZE_TL,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,
    seed=SEED,
)

print("Classes transfer learning :", train_ds_tl.class_names)

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

train_ds_tl = train_ds_tl.map(lambda images, labels: (preprocess_input(images), labels))
val_ds_tl = val_ds_tl.map(lambda images, labels: (preprocess_input(images), labels))

train_ds_tl = train_ds_tl.cache().prefetch(buffer_size=AUTOTUNE)
val_ds_tl = val_ds_tl.cache().prefetch(buffer_size=AUTOTUNE)

batch_images_tl, batch_labels_tl = next(iter(train_ds_tl))
print("Shape images TL batch :", batch_images_tl.shape)
print("Shape labels TL batch :", batch_labels_tl.shape)
print(f"Valeurs pixels TL : min={tf.reduce_min(batch_images_tl).numpy():.2f}, max={tf.reduce_max(batch_images_tl).numpy():.2f}")

assert batch_images_tl.shape == (BATCH_SIZE, IMG_SIZE_TL[0], IMG_SIZE_TL[1], 3)
assert batch_labels_tl.shape == (BATCH_SIZE, 1)
assert float(tf.reduce_min(batch_images_tl)) >= -1.0
assert float(tf.reduce_max(batch_images_tl)) <= 1.0

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE_TL[0], IMG_SIZE_TL[1], 3),
    include_top=False,
    weights="imagenet",
)

print("Nombre de couches MobileNetV2 :", len(base_model.layers))
print("Parametres totaux MobileNetV2 :", base_model.count_params())

base_model.trainable = False
print("Base MobileNetV2 trainable :", base_model.trainable)

In [ ]:
inputs = tf.keras.Input(shape=(IMG_SIZE_TL[0], IMG_SIZE_TL[1], 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model_tl = tf.keras.Model(inputs, outputs, name="mobilenetv2_transfer_learning")
model_tl.summary()

In [ ]:
trainable_params_tl = sum(tf.keras.backend.count_params(weight) for weight in model_tl.trainable_weights)
non_trainable_params_tl = sum(tf.keras.backend.count_params(weight) for weight in model_tl.non_trainable_weights)

print("Parametres entrainables TL     :", trainable_params_tl)
print("Parametres non entrainables TL :", non_trainable_params_tl)
print("Parametres base MobileNetV2    :", base_model.count_params())

assert model_tl.output_shape == (None, 1)
assert base_model.trainable is False
assert trainable_params_tl == 1280 * 128 + 128 + 128 * 1 + 1
assert non_trainable_params_tl == base_model.count_params()
assert model_tl.layers[-1].activation.__name__ == "sigmoid"

Verification attendue : `model_tl.summary()` doit montrer environ 2.2M de parametres non entrainables pour MobileNetV2 et environ 164k parametres entrainables pour la tete custom.

Edge case : sans `weights='imagenet'`, les poids sont aleatoires et le transfer learning perd son interet.

Adversarial : `include_top=True` garde la tete ImageNet a 1000 classes. Pour une classification binaire avec tete custom, `include_top=False` est obligatoire.

## Phase 3.2 - Entrainement de la tete uniquement

Objectif : entrainer uniquement la tete de classification pendant 10 epochs, observer une convergence rapide et verifier que la `val_accuracy` depasse 90%.

In [ ]:
assert base_model.trainable is False
assert all(layer.trainable is False for layer in base_model.layers)

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

print("Modele transfer learning compile avec base gelee.")

In [ ]:
log_dir_tl = "logs/transfer/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback_tl = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir_tl,
    histogram_freq=1,
)

print("TensorBoard log_dir transfer :", log_dir_tl)

In [ ]:
start = time.time()

history_tl_head = model_tl.fit(
    train_ds_tl,
    epochs=10,
    validation_data=val_ds_tl,
    callbacks=[tensorboard_callback_tl],
)

training_time_head = time.time() - start

print(f"Temps d'entrainement (tete seule) : {training_time_head:.0f}s")
print(f"val_accuracy finale (tete seule) : {max(history_tl_head.history['val_accuracy']):.3f}")

In [ ]:
%tensorboard --logdir logs/transfer

In [ ]:
plot_history(history_tl_head, "Transfer Learning - tete seule")

best_val_accuracy_tl_head = max(history_tl_head.history["val_accuracy"])
best_val_loss_tl_head = min(history_tl_head.history["val_loss"])
tl_head_gap = max(history_tl_head.history["accuracy"]) - best_val_accuracy_tl_head

print(f"Best val_accuracy TL tete seule : {best_val_accuracy_tl_head:.3f}")
print(f"Best val_loss TL tete seule     : {best_val_loss_tl_head:.4f}")
print(f"Gap train/val TL tete seule    : {tl_head_gap:.3f}")

if best_val_accuracy_tl_head >= 0.90:
    print("Objectif atteint : val_accuracy >= 90%.")
else:
    print("Objectif non atteint sur ce run : verifier dataset, preprocessing et courbes.")

Diagnostic attendu : la `val_accuracy` depasse souvent 90% en 3 a 5 epochs, avec une convergence plus rapide que TP1/TP2. Le gap train/val doit rester faible, car seule la tete custom apprend tandis que la base ImageNet reste gelee.

Edge case : avec `learning_rate=1e-1`, la tete bouge trop vite et la validation peut osciller violemment.

Adversarial : si `base_model.trainable = False` est oublie avant `compile`, toute la base MobileNetV2 s'entraine avec un learning rate trop eleve et les representations pre-entrainees peuvent se degrader.

## 3.4 - Tableau comparatif

A completer avec les metriques finales des trois modeles.

## 3.5 - Export ou exploration

A completer avec l'export `.tflite` et/ou une direction d'exploration.